In [12]:
import sys
sys.path.append(r'C:\Users\adeeb\OneDrive\Desktop\olist-ecommerce-analysis\src')

import sqlite3
import pandas as pd
from config import DB_PATH

conn = sqlite3.connect(DB_PATH)
pd.set_option('display.max_columns', None)

In [13]:
geo = pd.read_sql("SELECT * FROM geolocation", conn)

valid = (
    geo['geolocation_lat'].between(-34, 6) &
    geo['geolocation_lng'].between(-74, -28)
)

print("Rows outside Brazil's bounds:", (~valid).sum())
geo[~valid]['geolocation_city'].value_counts().head(10)

Rows outside Brazil's bounds: 31


geolocation_city
porto trombetas           5
santa rosa                3
santa maria               2
ilha dos valadares        2
santana do sobrado        1
santa lucia do piai       1
areia branca dos assis    1
varzea grande             1
vila dos cabanos          1
castelo dos sonhos        1
Name: count, dtype: int64

In [14]:
geo_clean = geo[valid]

dim_geography = (
    geo_clean
    .groupby('geolocation_zip_code_prefix')
    .agg(
        lat=('geolocation_lat', 'mean'),
        lng=('geolocation_lng', 'mean'),
        city=('geolocation_city', lambda s: s.mode().iloc[0]),
        state=('geolocation_state', lambda s: s.mode().iloc[0]),
    )
    .reset_index()
    .rename(columns={'geolocation_zip_code_prefix': 'zip_code_prefix'})
)

print("Unique zips in raw table:", geo['geolocation_zip_code_prefix'].nunique())
print("Rows in dim_geography:", len(dim_geography))
dim_geography.head()

Unique zips in raw table: 19015
Rows in dim_geography: 19011


,zip_code_prefix,lat,lng,city,state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


**The counts differ by 4 (19,015 raw vs 19,011), so 4 zip codes had only bad coordinates and dropped out entirely. That matters because any customer or seller in those zips would find no match in dim_geography when we join later.**

In [15]:
missing_zips = set(geo['geolocation_zip_code_prefix']) - set(dim_geography['zip_code_prefix'])
print("Zips lost:", missing_zips)
placeholders = ",".join(str(z) for z in missing_zips)

cust = pd.read_sql(
    f"SELECT COUNT(*) AS n FROM customers WHERE customer_zip_code_prefix IN ({placeholders})", conn)
sell = pd.read_sql(
    f"SELECT COUNT(*) AS n FROM sellers WHERE seller_zip_code_prefix IN ({placeholders})", conn)

print("Customers in those zips:", cust['n'][0])
print("Sellers in those zips:", sell['n'][0])
geo[geo['geolocation_zip_code_prefix'].isin(missing_zips)]


Zips lost: {18243, 78131, 83252, 95130}
Customers in those zips: 1
Sellers in those zips: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
822529,78131,38.816816,-9.394625,varzea grande,MT
860562,83252,42.184003,-8.723762,ilha dos valadares,PR
860832,83252,42.184003,-8.723762,ilha dos valadares,PR
965687,95130,14.585073,121.105394,santa lucia do piai,RS


In [16]:
dim_geography.to_sql('dim_geography', conn, if_exists='replace', index=False)    #saving dim_geography to the database
print("Rows saved:", len(dim_geography))

Rows saved: 19011


In [17]:
customers = pd.read_sql("SELECT * FROM customers", conn)

zips_per_person = customers.groupby('customer_unique_id')['customer_zip_code_prefix'].nunique()

print("People:", len(zips_per_person))
print("People with more than 1 zip:", (zips_per_person > 1).sum())

People: 96096
People with more than 1 zip: 250


In [18]:
dim_customer = (
    customers
    .drop_duplicates(subset='customer_unique_id')
    .rename(columns={
        'customer_zip_code_prefix': 'zip_code_prefix',
        'customer_city': 'city',
        'customer_state': 'state',
    })
    [['customer_unique_id', 'zip_code_prefix', 'city', 'state']]
)

dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
print("Rows in dim_customer:", len(dim_customer))

Rows in dim_customer: 96096


In [19]:
sellers = pd.read_sql("SELECT * FROM sellers", conn)

dim_seller = sellers.rename(columns={
    'seller_zip_code_prefix': 'zip_code_prefix',
    'seller_city': 'city',
    'seller_state': 'state',
})

dim_seller.to_sql('dim_seller', conn, if_exists='replace', index=False)
print("Rows in dim_seller:", len(dim_seller))
dim_seller.head()

Rows in dim_seller: 3095


,seller_id,zip_code_prefix,city,state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [23]:
products = pd.read_sql("SELECT * FROM products", conn)
translation = pd.read_sql("SELECT * FROM category_translation", conn) # need to attach English category names,since the raw categories are in Portuguese.

dim_product = products.merge(translation, on='product_category_name', how='left')

no_english = dim_product['product_category_name_english'].isna()
print("Products with no English name:", no_english.sum())



Products with no English name: 623


In [24]:
manual = {
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_and_food_preparers',
    'pc_gamer': 'pc_gamer',
}

dim_product['category_english'] = (
    dim_product['product_category_name_english']
    .fillna(dim_product['product_category_name'].map(manual))
    .fillna('unknown')
)

dim_product = dim_product.rename(columns={
    'product_category_name': 'category_pt',
    'product_name_lenght': 'name_length',
    'product_description_lenght': 'description_length',
    'product_photos_qty': 'photos_qty',
    'product_weight_g': 'weight_g',
    'product_length_cm': 'length_cm',
    'product_height_cm': 'height_cm',
    'product_width_cm': 'width_cm',
})

dim_product = dim_product[['product_id', 'category_pt', 'category_english', 'name_length',
                           'description_length', 'photos_qty', 'weight_g',
                           'length_cm', 'height_cm', 'width_cm']]

dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)
print("Rows in dim_product:", len(dim_product))
print("Still-empty English names:", dim_product['category_english'].isna().sum())

Rows in dim_product: 32951
Still-empty English names: 0


**- 13 products have categories missing from the translation table (10 portateis_cozinha_e_preparadores_de_alimentos, 3 pc_gamer). 
    Translated manually in dim_product. The 610 products with no category are labelled "unknown".**

In [25]:

orders = pd.read_sql("SELECT * FROM orders", conn)

date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

start = orders[date_cols].min().min().normalize()
end = orders[date_cols].max().max().normalize()
print("Calendar runs from", start.date(), "to", end.date())

dim_date = pd.DataFrame({'date': pd.date_range(start, end)})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['date'].dt.year
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name() 
dim_date['year_month'] = dim_date['date'].dt.strftime('%Y-%m')
dim_date['day'] = dim_date['date'].dt.day
dim_date['weekday_name'] = dim_date['date'].dt.day_name()
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek >= 5
dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')


dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
print("Rows in dim_date:", len(dim_date))
dim_date.head()

Calendar runs from 2016-09-04 to 2018-11-12
Rows in dim_date: 800


,date,date_key,year,quarter,month,month_name,year_month,day,weekday_name,is_weekend
0,2016-09-04,20160904,2016,3,9,September,2016-09,4,Sunday,True
1,2016-09-05,20160905,2016,3,9,September,2016-09,5,Monday,False
2,2016-09-06,20160906,2016,3,9,September,2016-09,6,Tuesday,False
3,2016-09-07,20160907,2016,3,9,September,2016-09,7,Wednesday,False
4,2016-09-08,20160908,2016,3,9,September,2016-09,8,Thursday,False


In [26]:
order_items = pd.read_sql("SELECT * FROM order_items", conn)

# Swap customer_id (one per order) for customer_unique_id (one per person)
orders_slim = orders.merge(
    customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

fact = order_items.merge(
    orders_slim[['order_id', 'customer_unique_id', 'order_status',
                 'order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date']],
    on='order_id', how='left')

def to_key(s):
    return pd.to_numeric(s.dt.strftime('%Y%m%d'), errors='coerce').astype('Int64')

fact['purchase_date_key'] = to_key(fact['order_purchase_timestamp'])
fact['delivered_date_key'] = to_key(fact['order_delivered_customer_date'])
fact['estimated_date_key'] = to_key(fact['order_estimated_delivery_date'])

fact['delivery_days'] = (fact['order_delivered_customer_date'] - fact['order_purchase_timestamp']).dt.days
fact['delay_days'] = (fact['order_delivered_customer_date'] - fact['order_estimated_delivery_date']).dt.days
fact['is_late'] = (fact['delay_days'] > 0).astype('Int64').where(fact['delay_days'].notna())

fact_order_items = fact[['order_id', 'order_item_id', 'product_id', 'seller_id',
                         'customer_unique_id', 'purchase_date_key', 'delivered_date_key',
                         'estimated_date_key', 'order_status', 'price', 'freight_value',
                         'delivery_days', 'delay_days', 'is_late']]

fact_order_items.to_sql('fact_order_items', conn, if_exists='replace', index=False)
print("Rows in fact_order_items:", len(fact_order_items))
print("Rows with no customer_unique_id:", fact_order_items['customer_unique_id'].isna().sum())

Rows in fact_order_items: 112650
Rows with no customer_unique_id: 0


In [27]:
fact

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,purchase_date_key,delivered_date_key,estimated_date_key,delivery_days,delay_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,871766c5855e863f6eccc05f988b23cb,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,20170913,20170920,20170929,7.0,-9.0,0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,eb28e67c4c0b83846050ddfb8a35d051,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,20170426,20170512,20170515,16.0,-3.0,0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,3818d81c6709e39d06b2738a8d3a2474,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,20180114,20180122,20180205,7.0,-14.0,0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,af861d436cfc08b2c2ddefd0ba074622,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,20180808,20180814,20180820,6.0,-6.0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,64b576fb70d441e8f1b2d7d446e483c5,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,20170204,20170301,20170317,25.0,-16.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,0c9aeda10a71f369396d0c04dce13a64,delivered,2018-04-23 13:57:06,2018-05-10 22:56:40,2018-05-18,20180423,20180510,20180518,17.0,-8.0,0
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,0da9fe112eae0c74d3ba1fe16de0988b,delivered,2018-07-14 10:26:46,2018-07-23 20:31:55,2018-08-01,20180714,20180723,20180801,9.0,-9.0,0
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,cd79b407828f02fdbba457111c38e4c4,delivered,2017-10-23 17:07:56,2017-10-28 12:22:22,2017-11-10,20171023,20171028,20171110,4.0,-13.0,0
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,eb803377c9315b564bdedad672039306,delivered,2017-08-14 23:02:59,2017-08-16 21:59:40,2017-08-25,20170814,20170816,20170825,1.0,-9.0,0


In [28]:
test = pd.Series(pd.to_datetime(['2017-11-28 17:44:07', None]))
print(to_key(test))

0    20171128
1        <NA>
dtype: Int64


In [29]:
x = pd.Series(['20171128', 'abc', None])
print(pd.to_numeric(x, errors='coerce'))

0    20171128.0
1           NaN
2           NaN
dtype: float64


In [32]:
payments = pd.read_sql("SELECT * FROM order_payments", conn)

fact_payments = payments[['order_id', 'payment_sequential', 'payment_type',
                          'payment_installments', 'payment_value']]

fact_payments.to_sql('fact_payments', conn, if_exists='replace', index=False)

print("Rows in fact_payments:", len(fact_payments))
print("Payments for orders not in orders table:",
      (~fact_payments['order_id'].isin(orders['order_id'])).sum())

Rows in fact_payments: 103886
Payments for orders not in orders table: 0


In [ ]:
checks = {
    'customer':       "LEFT JOIN dim_customer d ON f.customer_unique_id = d.customer_unique_id WHERE d.customer_unique_id IS NULL",
    'seller':         "LEFT JOIN dim_seller d ON f.seller_id = d.seller_id WHERE d.seller_id IS NULL",
    'product':        "LEFT JOIN dim_product d ON f.product_id = d.product_id WHERE d.product_id IS NULL",
    'purchase date':  "LEFT JOIN dim_date d ON f.purchase_date_key = d.date_key WHERE d.date_key IS NULL",
    'estimated date': "LEFT JOIN dim_date d ON f.estimated_date_key = d.date_key WHERE d.date_key IS NULL",
    'delivered date': "LEFT JOIN dim_date d ON f.delivered_date_key = d.date_key WHERE f.delivered_date_key IS NOT NULL AND d.date_key IS NULL",
}

print("Fact rows with no match in each dimension (all should be 0):")
for name, joins in checks.items():
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM fact_order_items f {joins}", conn)['n'][0]
    print(f"  {name:15s} {n}")

total = pd.read_sql("""
    SELECT COUNT(*) AS n FROM fact_order_items f
    LEFT JOIN dim_customer c ON f.customer_unique_id = c.customer_unique_id
    LEFT JOIN dim_seller   s ON f.seller_id          = s.seller_id
    LEFT JOIN dim_product  p ON f.product_id         = p.product_id
    LEFT JOIN dim_date     d ON f.purchase_date_key  = d.date_key
""", conn)['n'][0]
print("\nRows after joining all dimensions:", total, "(should equal the fact table, 112650)")

In [33]:
pd.read_sql("SELECT COUNT(*) AS n FROM fact_order_items", conn)

,n
0,112650
